# Question 5 - DCGAN Dimension Arithmetic (MNIST)
Transposed convolution output size:
$$H_{out} = (H_{in} - 1)\times s - 2p + k$$

- Step 1: Dense 100 -> 7x7x128, so the feature map is (128, 7, 7)
- **Step 2**: $(7-1)\times 2 - 2(1) + 4 = 14$, giving **(64, 14, 14)**, so **H, W = 14 x 14**
- **Step 3**: $(14-1)\times 2 - 2(1) + 4 = 28$, giving **(1, 28, 28)**, so **H, W = 28 x 28**

In Keras, `kernel_size=4, strides=2, padding='same'` gives the same result as PyTorch's `padding=1` (output = input x 2).

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

In [ ]:
def convT_out(h, k, s, p):
    return (h - 1) * s - 2 * p + k

h1 = convT_out(7, 4, 2, 1)
h2 = convT_out(h1, 4, 2, 1)
print("After step 2:", (h1, h1), " After step 3:", (h2, h2))

In [ ]:
class Generator(keras.Model):
    def __init__(self):
        super().__init__()
        self.fc = layers.Dense(7 * 7 * 128)
        self.reshape = layers.Reshape((7, 7, 128))
        self.up1 = layers.Conv2DTranspose(64, 4, strides=2, padding='same', activation='relu')
        self.up2 = layers.Conv2DTranspose(1, 4, strides=2, padding='same', activation='tanh')

    def call(self, z):
        x = self.reshape(self.fc(z));  print("after step 1:", x.shape)
        x = self.up1(x);               print("after step 2:", x.shape)
        x = self.up2(x);               print("after step 3:", x.shape)
        return x

G = Generator()
out = G(tf.random.normal((1, 100)))

In [ ]:
# Preprocessing for a Tanh output: scale MNIST to [-1, 1]
(x_train, _), _ = keras.datasets.mnist.load_data()
x_train = (x_train.astype("float32") - 127.5) / 127.5
print("pixel range:", x_train.min(), "to", x_train.max())

### 2. Pixel range
Tanh outputs values in **[-1, 1]**, so real MNIST images must be scaled to **[-1, 1]** with `x = (x - 127.5) / 127.5`.